In [4]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

# ======================================================
# Load data
# ======================================================
train_path = "C:/Users/aiden/School/6380/in-silico-drug-discovery/data/input_data.csv"
screen_path = "C:/Users/aiden/School/6380/in-silico-drug-discovery/data/list_of_compounds_for_computational_screening.csv"

df = pd.read_csv(train_path)
screen = pd.read_csv(screen_path)
research_preds = pd.read_csv("ResearchPaperPreds.csv")

# ======================================================
# Prepare training data
# ======================================================
labeled = df[df["senolytic"].isin([0, 1])].copy()

X = labeled.drop(columns=["senolytic", "ID", "SMILES"])
y = labeled["senolytic"].values

# Align screening features safely
X_screen = screen.reindex(columns=X.columns, fill_value=0)

# ======================================================
# Scale
# ======================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_screen_scaled = scaler.transform(X_screen)

# ======================================================
# Feature selection (top 160)
# ======================================================
rf = RandomForestClassifier(n_estimators=500, n_jobs=-1, class_weight="balanced")
rf.fit(X_scaled, y)

importances = rf.feature_importances_
feat_idx_sorted = np.argsort(importances)[::-1]

top_k = 160

X_selected = X_scaled[:, feat_idx_sorted[:top_k]]
X_screen_selected = X_screen_scaled[:, feat_idx_sorted[:top_k]]

# ======================================================
# Train FINAL model on FULL data
# ======================================================
sample_weights = np.where(y == 1, 27, 1)

model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss",
    objective="binary:logistic",
    max_depth=3,
    n_estimators=200,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
    n_jobs=-1
)

model.fit(X_selected, y, sample_weight=sample_weights)

# ======================================================
# Predict on screening set
# ======================================================
probs = model.predict_proba(X_screen_selected)[:, 1]

# 🔥 CUSTOM THRESHOLD
threshold = 0.46
preds = (probs >= threshold).astype(int)

# ======================================================
# Attach predictions
# ======================================================
screen["prediction_prob"] = probs
screen["pred"] = preds

# ======================================================
# Rank ONLY predicted positives
# ======================================================
predicted_positive = screen[screen["pred"] == 1].copy()

top21 = predicted_positive.sort_values(
    "prediction_prob", ascending=False
).head(21)

print("\nTOP 21 PREDICTIONS (XGBOOST):")
print(top21[["Name", "prediction_prob"]])

# ======================================================
# Cross-reference with research paper preds
# ======================================================
top21_names = set(top21["Name"].str.lower())
research_names = set(research_preds["Name"].str.lower())

overlap = top21_names.intersection(research_names)

overlap_df = top21[top21["Name"].str.lower().isin(overlap)]

print("\nOVERLAP WITH RESEARCH PAPER:")
print(overlap_df[["Name", "prediction_prob"]])

# ======================================================
# Save results
# ======================================================
top21.to_csv("Top21_XGBoost_Preds.csv", index=False)
overlap_df.to_csv("Overlap_XGBoost_Research.csv", index=False)


TOP 21 PREDICTIONS (XGBOOST):
                            Name  prediction_prob
2475                   Mangostin         0.976045
1754             gamma-Mangostin         0.975527
3590                  Selamectin         0.972909
3767         Sulconazole Nitrate         0.964241
3339                  Quinestrol         0.958696
712   BIX 01294 Trihydrochloride         0.951087
3083                   PF 573228         0.949912
1292                   Depofemin         0.940124
1791                    Geraniol         0.939749
3280                    Propofol         0.933239
1301                 Desogestrel         0.918862
3078                  Periplocin         0.917758
506                   Avobenzone         0.907494
2688                     Morusin         0.905247
2811                  Nimodipine         0.903394
306                Allylestrenol         0.897138
2818                 Nisoldipine         0.893624
952                    Celecoxib         0.893378
663                

In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

# ======================================================
# Load data
# ======================================================
train_path = "C:/Users/aiden/School/6380/in-silico-drug-discovery/data/input_data.csv"
screen_path = "C:/Users/aiden/School/6380/in-silico-drug-discovery/data/list_of_compounds_for_computational_screening.csv"

df = pd.read_csv(train_path)
screen = pd.read_csv(screen_path)
research_preds = pd.read_csv("ResearchPaperPreds.csv")

# ======================================================
# Prepare training data
# ======================================================
labeled = df[df["senolytic"].isin([0, 1])].copy()

X = labeled.drop(columns=["senolytic", "ID", "SMILES"])
y = labeled["senolytic"].values

# Align screening features
X_screen = screen.reindex(columns=X.columns, fill_value=0)

# ======================================================
# Scale
# ======================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_screen_scaled = scaler.transform(X_screen)

# ======================================================
# Feature selection (top 160)
# ======================================================
rf = RandomForestClassifier(n_estimators=500, n_jobs=-1, class_weight="balanced")
rf.fit(X_scaled, y)

importances = rf.feature_importances_
feat_idx_sorted = np.argsort(importances)[::-1]

top_k = 160

X_selected = X_scaled[:, feat_idx_sorted[:top_k]]
X_screen_selected = X_screen_scaled[:, feat_idx_sorted[:top_k]]

# ======================================================
# Train model
# ======================================================
sample_weights = np.where(y == 1, 27, 1)

model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss",
    objective="binary:logistic",
    max_depth=3,
    n_estimators=200,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
    n_jobs=-1
)

model.fit(X_selected, y, sample_weight=sample_weights)

# ======================================================
# Predict
# ======================================================
probs = model.predict_proba(X_screen_selected)[:, 1]

threshold = 0.46
preds = (probs >= threshold).astype(int)

screen["prediction_prob"] = probs
screen["pred"] = preds

# ======================================================
# Get predicted positives
# ======================================================
predicted_positive = screen[screen["pred"] == 1].copy()

print(f"\nTotal predicted positives: {len(predicted_positive)}")

# ======================================================
# Compare with research paper preds
# ======================================================
model_names = set(predicted_positive["Name"].str.lower())
research_names = set(research_preds["Name"].str.lower())

overlap = model_names.intersection(research_names)

overlap_df = predicted_positive[
    predicted_positive["Name"].str.lower().isin(overlap)
]

print(f"Overlap count: {len(overlap)}")

print("\nOVERLAPPING COMPOUNDS:")
print(overlap_df[["Name", "prediction_prob"]])

# ======================================================
# Save
# ======================================================
overlap_df.to_csv("Overlap_XGBoost_Research.csv", index=False)


Total predicted positives: 120
Overlap count: 12

OVERLAPPING COMPOUNDS:
                     Name  prediction_prob
741             BMS986142         0.833559
1604           Everolimus         0.704261
1754      gamma-Mangostin         0.962536
1755       Gamma-Oryzanol         0.509294
1948           Herbacetin         0.629424
2911            Oleandrin         0.820852
3078           Periplocin         0.958735
3370            Rapamycin         0.641277
3417        Ridaforolimus         0.741440
3583         Scutellarein         0.623471
4203  Vinblastine sulfate         0.526425
4332          Zotarolimus         0.482664
